In [1]:
# --- Cell 1: Import Libraries ---
import tensorflow as tf
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("Tất cả thư viện đã được import thành công!")

Tất cả thư viện đã được import thành công!


In [2]:
# --- Cell 2: Setup Data Paths ---
# Đường dẫn gốc của dự án
base_dir = r'E:\btlhttm\BTL-HTTM-2025'

# Đường dẫn đến thư mục dữ liệu cats_vs_dogs_small
data_dir = os.path.join(base_dir, 'cats_vs_dogs_small')
train_dir = os.path.join(data_dir, 'train')
validation_dir = os.path.join(data_dir, 'validation')

print(f"Thư mục training: {train_dir}")
print(f"Thư mục validation: {validation_dir}")

Thư mục training: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\train
Thư mục validation: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\validation


In [3]:
# --- Cell 3: Prepare Data Generators ---
# Thiết lập Data Augmentation cho dữ liệu training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

# Dữ liệu validation CHỈ rescale, không augment
validation_datagen = ImageDataGenerator(rescale=1./255)

# Tạo các generator để nạp dữ liệu
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [4]:
# --- Cell 4: Build the Model Architecture ---
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dropout(0.5), # Kỹ thuật chống overfitting có sẵn
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

C:\Users\FPT 2633\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# --- Cell 5: Define "Best Practices" Callbacks ---

# 1. EarlyStopping: Dừng sớm nếu không có cải thiện
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    restore_best_weights=True
)

# 2. ModelCheckpoint: Lưu lại phiên bản model tốt nhất
model_checkpoint = ModelCheckpoint(
    filepath='best_model_from_hung_kim.keras', # Tên file chứa model tốt nhất
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

# Tạo một danh sách chứa các callbacks sẽ sử dụng
callbacks_list = [early_stopping, model_checkpoint]

print("Callbacks 'EarlyStopping' và 'ModelCheckpoint' đã sẵn sàng!")

Callbacks 'EarlyStopping' và 'ModelCheckpoint' đã sẵn sàng!


In [6]:
# --- Cell 6: Train the Model ---
history = model.fit(
    train_generator,
    epochs=100, # Đặt số epoch lớn, EarlyStopping sẽ tự dừng khi cần
    validation_data=validation_generator,
    callbacks=callbacks_list # <- ÁP DỤNG CÁC BEST PRACTICES TẠI ĐÂY
)

C:\Users\FPT 2633\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 665ms/step - accuracy: 0.5315 - loss: 0.8700
Epoch 1: val_loss improved from None to 0.69217, saving model to best_model_from_hung_kim.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 57s 818ms/step - accuracy: 0.5010 - loss: 0.7463 - val_accuracy: 0.5200 - val_loss: 0.6922
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 762ms/step - accuracy: 0.5150 - loss: 0.6925
Epoch 2: val_loss did not improve from 0.69217
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 864ms/step - accuracy: 0.5075 - loss: 0.6936 - val_accuracy: 0.5370 - val_loss: 0.6925
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 752ms/step - accuracy: 0.4991 - loss: 0.6932
Epoch 3: val_loss improved from 0.69217 to 0.68797, saving model to best_model_from_hung_kim.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 55s 874ms/step - accuracy: 0.5080 - loss: 0.6928 - val_accuracy: 0.5510 - val_loss: 0.6880
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 682ms/step - accuracy: 0.5099 - loss: 0.6900
Epoch 4: val_loss improved from 0.68797 to 0.68758, savi

In [7]:
# --- Cell 7: Load and Verify the Best Model ---
print("\n--- Huấn luyện đã hoàn tất ---")

# Tải lại mô hình tốt nhất đã được lưu
best_model = tf.keras.models.load_model('best_model_from_hung_kim.keras')

print("\nĐã tải xong mô hình tốt nhất.")
print("Đây là mô hình nên được sử dụng để đánh giá trên tập test.")

# In ra cấu trúc của mô hình tốt nhất để xác nhận
best_model.summary()


--- Huấn luyện đã hoàn tất ---

Đã tải xong mô hình tốt nhất.
Đây là mô hình nên được sử dụng để đánh giá trên tập test.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 57,102,533 (217.83 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 38,068,356 (145.22 MB)

In [8]:
# --- Cell 8: Evaluate the Best Model on the Test Set ---

# Đường dẫn đến thư mục test
test_dir = os.path.join(data_dir, 'test')
print(f"Thư mục test: {test_dir}")

# Tạo generator cho dữ liệu test (CHỈ RESCALE, KHÔNG AUGMENT)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    shuffle=False # Không cần xáo trộn dữ liệu test
)

# Dùng mô hình tốt nhất để đánh giá
print("\nĐang tiến hành đánh giá mô hình trên tập Test...")
test_loss, test_accuracy = best_model.evaluate(test_generator)

print(f"\nKết quả cuối cùng trên tập Test:")
print(f"   - Mất mát (Loss): {test_loss:.4f}")
print(f"   - Độ chính xác (Accuracy): {test_accuracy*100:.2f}%")

Thư mục test: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\test
Found 2000 images belonging to 2 classes.

Đang tiến hành đánh giá mô hình trên tập Test...
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 163ms/step - accuracy: 0.7515 - loss: 0.5034

Kết quả cuối cùng trên tập Test:
   - Mất mát (Loss): 0.5034
   - Độ chính xác (Accuracy): 75.15%
